# 02 – Data Cleaning
Cleans the SIMD 2020v2 indicators and joins the Data Zone (2011) Urban Rural Classification 2020,
based on findings from `01_data_exploration.ipynb`. Outputs are saved to `data/processed/`.

## 1. Load data and select columns
Keep identifiers, population, income deprivation and the nine travel time columns.
Health, education, crime and housing indicators are outside the scope of this project.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

simd_file = next(RAW.glob("*indicators*.xlsx"))
ur_file = next(RAW.glob("datazone2011_urban_rural*.csv"))

simd = pd.read_excel(simd_file, sheet_name="Data")
ur = pd.read_csv(ur_file)

In [2]:
travel_cols = [c for c in simd.columns if c.startswith(("drive_", "PT_"))]
keep = ["Data_Zone", "Intermediate_Zone", "Council_area", "Total_population", "Income_rate"] + travel_cols

simd = simd[keep]
print(simd.shape)

(6976, 14)


In [3]:
as_number = pd.to_numeric(simd["Income_rate"], errors="coerce")
not_numeric = simd.loc[as_number.isna(), "Income_rate"]

print("Non-numeric values:", len(not_numeric))
print(not_numeric.value_counts())

Non-numeric values: 3
Income_rate
*    3
Name: count, dtype: int64


**Result:** `Income_rate` contains 3 suppressed values ("*"). These are converted to missing
so the column becomes numeric. The 3 zones stay in the travel analysis and are excluded only
from deprivation comparisons.

In [4]:
simd["Income_rate"] = pd.to_numeric(simd["Income_rate"], errors="coerce")

print(simd["Income_rate"].dtype, "| missing:", simd["Income_rate"].isna().sum())
simd["Income_rate"].describe()

float64 | missing: 3


count    6973.000000
mean        0.122900
std         0.096091
min         0.000000
25%         0.040000
50%         0.100000
75%         0.180000
max         0.590000
Name: Income_rate, dtype: float64

## 2. Add the Urban Rural classification
Join the 6-fold classification by data zone, add readable labels, and create a simpler
3-group version (Urban / Small Towns / Rural) for clearer comparisons.

In [5]:
rows_before = len(simd)

simd = simd.merge(
    ur[["DataZone", "UrbanRural6fold2020"]],
    left_on="Data_Zone", right_on="DataZone",
    how="left", validate="one_to_one"
).drop(columns="DataZone")

print("Rows:", rows_before, "->", len(simd))
print("Missing classification:", simd["UrbanRural6fold2020"].isna().sum())

Rows: 6976 -> 6976
Missing classification: 0


In [6]:
ur_labels = {1: "Large Urban Areas", 2: "Other Urban Areas", 3: "Accessible Small Towns",
             4: "Remote Small Towns", 5: "Accessible Rural", 6: "Remote Rural"}
ur_groups = {1: "Urban", 2: "Urban", 3: "Small Towns", 4: "Small Towns", 5: "Rural", 6: "Rural"}

simd["UR_class"] = simd["UrbanRural6fold2020"].map(ur_labels)
simd["UR_group"] = simd["UrbanRural6fold2020"].map(ur_groups)

simd.groupby(["UrbanRural6fold2020", "UR_class", "UR_group"]).size()

UrbanRural6fold2020  UR_class                UR_group   
1                    Large Urban Areas       Urban          2496
2                    Other Urban Areas       Urban          2459
3                    Accessible Small Towns  Small Towns     613
4                    Remote Small Towns      Small Towns     202
5                    Accessible Rural        Rural           805
6                    Remote Rural            Rural           401
dtype: int64

**Result:** all 6,976 zones matched (one-to-one join, no rows lost). Each code maps to the
correct label and group, with counts identical to exploration.